In [0]:
from pyspark.sql import functions as F
import logging
import sys

logger = logging.getLogger("turbines")
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)
logger.propagate = False

In [0]:
def gap(df):
    target = (
        (F.col("turbine_id") == 7) &
        (F.to_date("timestamp") == "2022-03-15") &
        (F.hour("timestamp") < 6)
    )
    n = df.filter(target).count()
    logger.info(f"delete_gap: removing {n} rows (turbine 7, 2022-03-15 00:00-05:00)")
    return df.filter(~target)


def inject_outliers(df):
    t1 = (F.col("turbine_id") == 2)  & (F.col("timestamp") == "2022-03-08 14:00:00")   
    t2 = (F.col("turbine_id") == 9)  & (F.col("timestamp") == "2022-03-12 03:00:00")   
    t3 = (F.col("turbine_id") == 12) & (F.col("timestamp") == "2022-03-20 17:00:00")  
    out = df.withColumn("power_output",
        F.when(t1, F.lit(999.0))
         .when(t2, F.lit(-5.0))
         .when(t3, F.lit(6.8))
         .otherwise(F.col("power_output")))
    logger.info("inject_outliers: 3 rows set to 999 / -5 / 6.8")
    return out


def inject_stuck(df):
    target = (F.col("turbine_id") == 3) & (F.to_date("timestamp") == "2022-03-25")
    n = df.filter(target).count()
    logger.info(f"inject_stuck: turbine 3 flatlined at 0.2 for {n} rows on 2022-03-25")
    return df.withColumn("power_output",
        F.when(target, F.lit(0.2)).otherwise(F.col("power_output")))
    
def inject_nulls(df):
    target = (
        (F.col("turbine_id") == 4) &
        (F.col("timestamp").between("2022-03-10 08:00:00", "2022-03-10 12:00:00"))
    )
    n = df.filter(target).count()
    logger.info(f"inject_nulls: nulling power_output on {n} rows (turbine 4, 2022-03-10 08:00-12:00)")
    return df.withColumn(
        "power_output",
        F.when(target, None).otherwise(F.col("power_output"))
    )

In [0]:
storage_account = "sacuksnprdcdproject0001"

path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/project0001/nprd/turbine/landing/*.csv"

df = spark.read.option("header", True).csv(path)
logger.info(f"Read {df.count()} records from {path}")

In [0]:
df_casted = (
    df.withColumn("timestamp", F.col("timestamp").cast("timestamp"))
    .withColumn("turbine_id", F.col("turbine_id").cast("int"))
    .withColumn("wind_speed", F.col("wind_speed").cast("float"))
    .withColumn("wind_direction", F.col("wind_direction").cast("float"))
    .withColumn("power_output", F.col("power_output").cast("float"))
)

In [0]:
df_corrupted = gap(df_casted)              
df_corrupted = inject_nulls(df_corrupted)    
df_corrupted = inject_outliers(df_corrupted)  
df_corrupted = inject_stuck(df_corrupted)     

In [0]:
corrupted_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/project0001/nprd/turbine/landing_corrupted/"

(df_corrupted
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(corrupted_path))

logger.info(f"Wrote corrupted dataset to {corrupted_path}")

## Check its worked

In [0]:
check = (spark.read.option("header", True).csv(corrupted_path)
    .withColumn("power_output", F.col("power_output").cast("float"))
    .withColumn("timestamp", F.col("timestamp").cast("timestamp")))

print("total rows:", check.count())                                        
print("nulls:", check.filter(F.col("power_output").isNull()).count())      
check.filter((F.col("power_output") < 0) | (F.col("power_output") > 5)).show()   

In [0]:
(check.groupBy(F.to_date("timestamp").alias("d"), "turbine_id").count()
    .filter("count < 24").show())     # expect turbine 7 on 03-15 with 18